In [4]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [easyocr]m4/5 [easyocr]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [14]:
from ultralytics import YOLO
import easyocr
import re

# 1) Load YOLO license-plate detector and OCR model
model = YOLO("saved_models/license_plate_best.pt")  # path to your fine-tuned weights
reader = easyocr.Reader(['en'], gpu=True)

# 2) Regex for plate pattern (example: XX11XXX)
plate_pattern = re.compile(r"[A-Z]{2}[0-9]{2}[A-Z]{3}")
CONF_THRESH = 0.3


In [26]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VO6n0TiuYhQiosuP3A5F")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(11)
dataset = version.download("coco")
                


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
loading Roboflow workspace...
loading Roboflow project...


In [32]:
import torch

if torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

Using device: mps


In [34]:
# Plate-format correction helpers
mapping_num_to_alpha = {
    "0": "O", "1": "I", "2": "Z", "3": "B", "5": "S", "6": "G", "8": "B"
}
mapping_alpha_to_num = {
    "O": "0", "Q": "0", "I": "1", "Z": "2", "B": "8", "S": "5"
}

def correct_plate_format(ocr_text: str) -> str:
    """
    Example rule-based cleanup for format: LLDDLLL (L=letter, D=digit).
    Tune for your exact pattern.
    """
    if not ocr_text:
        return ""

    ocr_text = ocr_text.upper().replace(" ", "")
    if len(ocr_text) != 7:
        return ""

    corrected = []

    for i, ch in enumerate(ocr_text):
        # positions 0,1,4,5,6 -> letters
        if i in [0, 1, 4, 5, 6]:
            if ch.isdigit() and ch in mapping_num_to_alpha:
                corrected.append(mapping_num_to_alpha[ch])
            elif ch.isalpha():
                corrected.append(ch)
            else:
                return ""
        # positions 2,3 -> digits
        else:
            if ch.isalpha() and ch in mapping_alpha_to_num:
                corrected.append(mapping_alpha_to_num[ch])
            elif ch.isdigit():
                corrected.append(ch)
            else:
                return ""

    candidate = "".join(corrected)
    if not plate_pattern.fullmatch(candidate):
        return ""
    return candidate


In [36]:
# Temporal stabilization buffers
plate_history = defaultdict(lambda: deque(maxlen=10))  # last 10 predictions per box
plate_final = {}                                       # most stable text per box

def get_box_id(x1, y1, x2, y2) -> int:
    # Simple pseudo-ID using coarse coordinates
    return int(x1 / 10) * 1000000 + int(y1 / 10) * 10000 + int(x2 / 10) * 100 + int(y2 / 10)

def get_stable_plate(box_id, new_text: str) -> str:
    if new_text:
        plate_history[box_id].append(new_text)
        most_common = max(set(plate_history[box_id]), key=plate_history[box_id].count)
        plate_final[box_id] = most_common
    return plate_final.get(box_id, "")


In [38]:
# Temporal stabilization buffers
plate_history = defaultdict(lambda: deque(maxlen=10))  # last 10 predictions per box
plate_final = {}                                       # most stable text per box

def get_box_id(x1, y1, x2, y2) -> int:
    # Simple pseudo-ID using coarse coordinates
    return int(x1 / 10) * 1000000 + int(y1 / 10) * 10000 + int(x2 / 10) * 100 + int(y2 / 10)

def get_stable_plate(box_id, new_text: str) -> str:
    if new_text:
        plate_history[box_id].append(new_text)
        most_common = max(set(plate_history[box_id]), key=plate_history[box_id].count)
        plate_final[box_id] = most_common
    return plate_final.get(box_id, "")


In [40]:
def recognize_plate(plate_crop: np.ndarray) -> str:
    """
    Runs EasyOCR on plate crop and applies format correction + regex check.
    """
    if plate_crop.size == 0:
        return ""

    result = reader.readtext(plate_crop, detail=0, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    if not result:
        return ""

    raw = result[0]
    candidate = correct_plate_format(raw)
    if candidate and plate_pattern.fullmatch(candidate):
        return candidate
    return ""


In [54]:
# Video input/output
input_video = "/Users/aryan/Desktop/computer_vision/hands_on_cv/traffic_video1.mp4"          # your source
output_video = "annotated_output.mp4"

cap = cv2.VideoCapture(input_video)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_video, fourcc, fps, (w, h))


In [56]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    for r in results:
        boxes = r.boxes
        for box in boxes:
            conf = float(box.conf.cpu().numpy()[0])
            if conf < CONF_THRESH:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            x1 = max(0, x1); y1 = max(0, y1)
            x2 = min(frame.shape[1], x2); y2 = min(frame.shape[0], y2)

            plate_crop = frame[y1:y2, x1:x2]

            # OCR with correction
            text = recognize_plate(plate_crop)

            # Stabilize using history
            box_id = get_box_id(x1, y1, x2, y2)
            stable_text = get_stable_plate(box_id, text)

            # Draw rectangle around plate
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

            # Overlay zoomed-in plate above box
            if plate_crop.size > 0:
                overlay_h, overlay_w = 150, 400
                plate_resized = cv2.resize(plate_crop, (overlay_w, overlay_h))

                oy1 = max(0, y1 - overlay_h - 40)
                ox1 = x1
                oy2 = oy1 + overlay_h
                ox2 = ox1 + overlay_w

                if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
                    frame[oy1:oy2, ox1:ox2] = plate_resized

                    # Show stabilized OCR text above overlay
                    if stable_text:
                        # black outline
                        cv2.putText(frame, stable_text, (ox1, oy1 - 20),
                                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 6)
                        # white text
                        cv2.putText(frame, stable_text, (ox1, oy1 - 20),
                                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)

    out.write(frame)
    cv2.imshow("Annotated Video", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print("Annotated video saved as", output_video)


Annotated video saved as annotated_output.mp4
